In [ ]:
# Import required libraries
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json
from collections import defaultdict
import shutil

In [ ]:
# Install required packages if not already installed
# !pip install opencv-python mtcnn facenet-pytorch pillow tqdm pandas

In [ ]:
# Configuration
DATASET_ROOT = Path('celebdfv2')
OUTPUT_ROOT = Path('celebdfv2_images')
TEST_LIST_FILE = DATASET_ROOT / 'List_of_testing_videos.txt'

# Preprocessing parameters
FRAMES_PER_VIDEO = 30  # Number of frames to extract per video
FACE_SIZE = (224, 224)  # Size to resize faces
FACE_MARGIN = 0.3  # Margin around detected face (30%)
MIN_FACE_SIZE = 80  # Minimum face size to accept (in pixels)
DETECTION_CONFIDENCE = 0.9  # Face detection confidence threshold

# Create output directories
OUTPUT_ROOT.mkdir(exist_ok=True)
(OUTPUT_ROOT / 'train' / 'real').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'train' / 'fake').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'test' / 'real').mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT / 'test' / 'fake').mkdir(parents=True, exist_ok=True)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Frames per video: {FRAMES_PER_VIDEO}")
print(f"Face size: {FACE_SIZE}")

## 1. Load Test Split Information

In [ ]:
# Load test video list
test_videos = set()
if TEST_LIST_FILE.exists():
    with open(TEST_LIST_FILE, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                # Format: label path
                video_path = parts[1]
                test_videos.add(video_path)
    print(f"Loaded {len(test_videos)} test videos")
else:
    print("Test list file not found. Will create random split.")

# Display some examples
print("\nExample test videos:")
for i, vid in enumerate(list(test_videos)[:5]):
    print(f"  {vid}")

## 2. Initialize Face Detector

We'll use MTCNN for robust face detection. Alternative: dlib or OpenCV's Haar cascades.

In [ ]:
# Try to import MTCNN, fallback to OpenCV if not available
try:
    from facenet_pytorch import MTCNN
    import torch
    
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    mtcnn = MTCNN(
        image_size=FACE_SIZE[0],
        margin=0,
        min_face_size=MIN_FACE_SIZE,
        thresholds=[0.6, 0.7, DETECTION_CONFIDENCE],
        factor=0.709,
        post_process=False,
        device=device,
        keep_all=False  # Only keep the most confident face
    )
    USE_MTCNN = True
    print("✓ Using MTCNN for face detection")
    
except ImportError:
    print("MTCNN not available, using OpenCV Haar Cascade")
    USE_MTCNN = False
    # Load OpenCV face detector
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    print("✓ Using OpenCV Haar Cascade for face detection")

## 3. Face Detection and Cropping Functions

In [ ]:
def detect_face_mtcnn(frame, mtcnn):
    """Detect face using MTCNN and return bounding box."""
    try:
        # MTCNN expects RGB
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes, probs = mtcnn.detect(rgb_frame)
        
        if boxes is not None and len(boxes) > 0:
            # Get the most confident detection
            best_idx = np.argmax(probs)
            box = boxes[best_idx]
            return box, probs[best_idx]
    except Exception as e:
        pass
    return None, None


def detect_face_opencv(frame, face_cascade):
    """Detect face using OpenCV Haar Cascade and return bounding box."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, 
        scaleFactor=1.1, 
        minNeighbors=5, 
        minSize=(MIN_FACE_SIZE, MIN_FACE_SIZE)
    )
    
    if len(faces) > 0:
        # Get the largest face
        areas = [w * h for (x, y, w, h) in faces]
        best_idx = np.argmax(areas)
        x, y, w, h = faces[best_idx]
        # Convert to [x1, y1, x2, y2] format
        return [x, y, x + w, y + h], 1.0
    return None, None


def crop_face_with_margin(frame, box, margin=0.3):
    """Crop face from frame with margin and resize."""
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = map(int, box)
    
    # Add margin
    face_w = x2 - x1
    face_h = y2 - y1
    margin_w = int(face_w * margin)
    margin_h = int(face_h * margin)
    
    # Ensure we stay within frame bounds
    x1 = max(0, x1 - margin_w)
    y1 = max(0, y1 - margin_h)
    x2 = min(w, x2 + margin_w)
    y2 = min(h, y2 + margin_h)
    
    # Crop and resize
    face_img = frame[y1:y2, x1:x2]
    
    if face_img.size == 0:
        return None
        
    face_img = cv2.resize(face_img, FACE_SIZE, interpolation=cv2.INTER_AREA)
    return face_img


def extract_frames_from_video(video_path, num_frames=30):
    """Extract evenly spaced frames from video."""
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames < num_frames:
        # Use all frames if video is short
        frame_indices = list(range(total_frames))
    else:
        # Sample evenly spaced frames
        frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    
    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
    
    cap.release()
    return frames


print("✓ Face detection and cropping functions defined")

## 4. Video Processing Function

In [ ]:
def process_video(video_path, output_dir, video_id, label):
    """Process a single video: extract frames, detect faces, and save."""
    try:
        # Extract frames
        frames = extract_frames_from_video(video_path, FRAMES_PER_VIDEO)
        
        if len(frames) == 0:
            return 0, "No frames extracted"
        
        saved_count = 0
        
        for frame_idx, frame in enumerate(frames):
            # Detect face
            if USE_MTCNN:
                box, conf = detect_face_mtcnn(frame, mtcnn)
            else:
                box, conf = detect_face_opencv(frame, face_cascade)
            
            if box is None:
                continue
            
            # Crop face
            face_img = crop_face_with_margin(frame, box, FACE_MARGIN)
            
            if face_img is None:
                continue
            
            # Save image
            output_path = output_dir / f"{video_id}_frame{frame_idx:04d}.jpg"
            cv2.imwrite(str(output_path), face_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            saved_count += 1
        
        return saved_count, "Success"
    
    except Exception as e:
        return 0, f"Error: {str(e)}"


print("✓ Video processing function defined")

## 5. Organize Videos by Category and Split

In [ ]:
# Organize videos by category
video_categories = {
    'real': [
        ('Celeb-real', 'real'),
        ('YouTube-real', 'real'),
    ],
    'fake': [
        ('Celeb-synthesis', 'fake'),
    ]
}

# Collect all videos
all_videos = []

for label_type, folders in video_categories.items():
    for folder_name, label in folders:
        folder_path = DATASET_ROOT / folder_name
        if not folder_path.exists():
            print(f"Warning: {folder_path} not found")
            continue
        
        video_files = list(folder_path.glob('*.mp4'))
        print(f"Found {len(video_files)} videos in {folder_name}")
        
        for video_file in video_files:
            # Determine split
            relative_path = f"{folder_name}/{video_file.name}"
            is_test = relative_path in test_videos
            split = 'test' if is_test else 'train'
            
            all_videos.append({
                'path': video_file,
                'label': label,
                'split': split,
                'folder': folder_name,
                'video_id': video_file.stem
            })

print(f"\nTotal videos to process: {len(all_videos)}")

# Print statistics
stats = defaultdict(lambda: defaultdict(int))
for video in all_videos:
    stats[video['split']][video['label']] += 1

print("\nDataset statistics:")
for split in ['train', 'test']:
    print(f"  {split.capitalize()}:")
    for label in ['real', 'fake']:
        print(f"    {label}: {stats[split][label]} videos")

## 6. Process All Videos

In [ ]:
# Process all videos
results = []
failed_videos = []

print("Starting video processing...\n")

for video_info in tqdm(all_videos, desc="Processing videos"):
    video_path = video_info['path']
    label = video_info['label']
    split = video_info['split']
    video_id = f"{video_info['folder']}_{video_info['video_id']}"
    
    # Determine output directory
    output_dir = OUTPUT_ROOT / split / label
    
    # Process video
    saved_count, status = process_video(video_path, output_dir, video_id, label)
    
    result = {
        'video_path': str(video_path),
        'video_id': video_id,
        'label': label,
        'split': split,
        'frames_saved': saved_count,
        'status': status
    }
    results.append(result)
    
    if saved_count == 0:
        failed_videos.append(result)

print("\n✓ Processing complete!")

## 7. Save Processing Results and Statistics

In [ ]:
# Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_ROOT / 'processing_results.csv', index=False)
print(f"✓ Saved processing results to {OUTPUT_ROOT / 'processing_results.csv'}")

# Save failed videos
if failed_videos:
    failed_df = pd.DataFrame(failed_videos)
    failed_df.to_csv(OUTPUT_ROOT / 'failed_videos.csv', index=False)
    print(f"✓ Saved {len(failed_videos)} failed videos to {OUTPUT_ROOT / 'failed_videos.csv'}")

# Calculate and save statistics
stats = {
    'total_videos_processed': len(results),
    'successful_videos': len([r for r in results if r['frames_saved'] > 0]),
    'failed_videos': len(failed_videos),
    'total_frames_extracted': sum(r['frames_saved'] for r in results),
    'split_statistics': {}
}

for split in ['train', 'test']:
    split_results = [r for r in results if r['split'] == split]
    stats['split_statistics'][split] = {
        'total_videos': len(split_results),
        'real_videos': len([r for r in split_results if r['label'] == 'real']),
        'fake_videos': len([r for r in split_results if r['label'] == 'fake']),
        'real_frames': sum(r['frames_saved'] for r in split_results if r['label'] == 'real'),
        'fake_frames': sum(r['frames_saved'] for r in split_results if r['label'] == 'fake'),
    }

# Save statistics
with open(OUTPUT_ROOT / 'dataset_statistics.json', 'w') as f:
    json.dump(stats, f, indent=2)

print(f"\n✓ Saved statistics to {OUTPUT_ROOT / 'dataset_statistics.json'}")

## 8. Display Final Statistics

In [ ]:
# Print summary
print("\n" + "="*60)
print("DATASET PREPROCESSING SUMMARY")
print("="*60)
print(f"\nTotal videos processed: {stats['total_videos_processed']}")
print(f"Successful: {stats['successful_videos']}")
print(f"Failed: {stats['failed_videos']}")
print(f"Total frames extracted: {stats['total_frames_extracted']:,}")

for split in ['train', 'test']:
    split_stats = stats['split_statistics'][split]
    print(f"\n{split.upper()} SET:")
    print(f"  Total videos: {split_stats['total_videos']}")
    print(f"  Real videos: {split_stats['real_videos']} ({split_stats['real_frames']:,} frames)")
    print(f"  Fake videos: {split_stats['fake_videos']} ({split_stats['fake_frames']:,} frames)")

print("\n" + "="*60)
print(f"Output directory: {OUTPUT_ROOT}")
print("Directory structure:")
print("  celebdfv2_images/")
print("  ├── train/")
print("  │   ├── real/")
print("  │   └── fake/")
print("  ├── test/")
print("  │       ├── real/")
print("  │       └── fake/")
print("  ├── processing_results.csv")
print("  ├── dataset_statistics.json")
print("  └── failed_videos.csv (if any)")
print("="*60)

## 9. Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt

def show_sample_images(split='train', num_samples=8):
    """Display sample images from the dataset."""
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f'Sample Images from {split.upper()} Set', fontsize=16)
    
    for idx, label in enumerate(['real', 'fake']):
        image_dir = OUTPUT_ROOT / split / label
        image_files = list(image_dir.glob('*.jpg'))[:4]
        
        for i, img_path in enumerate(image_files):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            axes[idx, i].imshow(img)
            axes[idx, i].set_title(f"{label.upper()}: {img_path.stem}")
            axes[idx, i].axis('off')
    
    plt.tight_layout()
    plt.savefig(OUTPUT_ROOT / f'sample_{split}_images.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Saved sample visualization to {OUTPUT_ROOT / f'sample_{split}_images.png'}")

# Show samples from both splits
show_sample_images('train')
show_sample_images('test')

## 10. Create Data Loading Script (Optional)

In [ ]:
# Create a simple data loading script for PyTorch
dataloader_code = '''import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path

class CelebDFImageDataset(Dataset):
    """CelebDF-v2 Image Dataset for PyTorch."""
    
    def __init__(self, root_dir, split='train', transform=None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.transform = transform
        
        # Collect all image paths
        self.samples = []
        for label_idx, label in enumerate(['real', 'fake']):
            label_dir = self.root_dir / split / label
            for img_path in label_dir.glob('*.jpg'):
                self.samples.append((str(img_path), label_idx))
        
        print(f"Loaded {len(self.samples)} images from {split} set")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Example usage:
if __name__ == "__main__":
    # Define transforms
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = CelebDFImageDataset('celebdfv2_images', split='train', transform=train_transform)
    test_dataset = CelebDFImageDataset('celebdfv2_images', split='test', transform=test_transform)
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
    
    print(f"Train batches: {len(train_loader)}")
    print(f"Test batches: {len(test_loader)}")
'''

# Save the dataloader script
with open('celebdfv2_dataloader.py', 'w') as f:
    f.write(dataloader_code)

print("✓ Created celebdfv2_dataloader.py for PyTorch data loading")

## Done!

The CelebDF-v2 dataset has been successfully converted to an image dataset.

### Next Steps:
1. Use the `celebdfv2_dataloader.py` script to load data in your training pipeline
2. Train your deepfake detection model (CNN, Vision Transformer, etc.)
3. Evaluate on the test set

### Tips:
- Consider data augmentation for better generalization
- Try different architectures (EfficientNet, ResNet, Vision Transformer)
- Experiment with different face detection methods for better results
- Consider using temporal information if needed (extract sequences instead of individual frames)